# Limpieza de datos

### Importación de librerías

In [21]:
import json
import pandas as pd
from huggingface_hub import hf_hub_download
from pathlib import Path

### Rutas

In [22]:
repo_id = 'dreamerdeo/multispider'
carpeta = 'dataset/multispider/with_english_value'
carpeta_esquemas = Path('../esquemas')
datos_crudos = '../datos_crudos/'
datos_limpios = '../datos/'

### Importación de datos

In [23]:
# Datos del LLM
df_llm = pd.read_csv(f'{datos_crudos}/datos_llm.csv')
df_llm.head(2)

,id,esquema_id,nivel,esquema,pregunta,sql
0,comercio_001,comercio,facil,CREATE TABLE `clientes` (`id_cliente` INT NOT ...,"Muestra el identificador, nombre y correo de c...","SELECT p.`id_cliente`, p.`nombre`, p.`correo` ..."
1,comercio_002,comercio,facil,CREATE TABLE `clientes` (`id_cliente` INT NOT ...,¿Qué clientes viven en Guatemala? Devuelve ide...,"SELECT p.`id_cliente`, p.`nombre`, p.`ciudad` ..."


In [24]:
# Datos de Multi-Spider
ruta_preguntas = hf_hub_download(repo_id=repo_id,
                                 filename=f'{carpeta}/train_es.json',
                                 repo_type='dataset')

# Se cargan los datos en un json
with open(ruta_preguntas, "r", encoding="utf-8") as f:
    preguntas = json.load(f)

### Descarga de los esquemas

In [31]:
# Esquemas de Multi-Spider
ruta_esquemas = hf_hub_download(repo_id=repo_id,
                                filename=f'{carpeta}/tables_es.json',
                                repo_type='dataset')

# Se cargan los datos en un json
with open(ruta_esquemas, "r", encoding="utf-8") as f:
    esquemas = json.load(f)

# Tipos de datos
tipos = {'text': 'VARCHAR(100)',
         'number': 'DECIMAL(18,2)',
         'time': 'DATETIME',
         'boolean': 'BOOLEAN'}

for esquema in esquemas:

    db_id = esquema['db_id']

    with open(carpeta_esquemas / f"{db_id}.sql", "w", encoding="utf-8") as f:

        for i, tabla in enumerate(esquema["table_names_original"]):

            columnas = [
                f"`{nombre}` {tipos.get(esquema['column_types'][j], 'VARCHAR(100)')}"
                for j, (indice, nombre) in enumerate(esquema["column_names_original"])
                if indice == i and nombre != "*"
            ]

            f.write(f"CREATE TABLE `{tabla}` (")
            f.write(", ".join(columnas))
            f.write(");\n")

print(f"Se guardaron {len(esquemas)} archivos SQL.")

Se guardaron 166 archivos SQL.


### Conversión de archivos en línea

In [26]:
df_spider = pd.DataFrame([{'db_id': registro['db_id'],
                           'pregunta': registro['question'],
                           'sql': registro['query']} for registro in preguntas])
df_spider['sql'] = df_spider['sql'].str.replace(' ,', ',')
df_spider.head(2)

,db_id,pregunta,sql
0,department_management,¿Cuántos jefes de departamento tienen más de 5...,SELECT count(*) FROM head WHERE age > 56
1,department_management,"Indique el nombre, estado de nacimiento y edad...","SELECT name, born_state, age FROM head ORDER..."


### Homologando los datos del LLM

In [27]:
df_llm = df_llm[['esquema_id', 'pregunta', 'sql']]
df_llm['sql'] = df_llm['sql'].str.replace('`', '')
df_llm.columns = df_spider.columns
df_llm.head(2)

,db_id,pregunta,sql
0,comercio,"Muestra el identificador, nombre y correo de c...","SELECT p.id_cliente, p.nombre, p.correo FROM c..."
1,comercio,¿Qué clientes viven en Guatemala? Devuelve ide...,"SELECT p.id_cliente, p.nombre, p.ciudad FROM c..."


### Unión de datos de entrenamiento

In [28]:
df = pd.concat([df_spider, df_llm]).reset_index(drop=True)
df.head()

,db_id,pregunta,sql
0,department_management,¿Cuántos jefes de departamento tienen más de 5...,SELECT count(*) FROM head WHERE age > 56
1,department_management,"Indique el nombre, estado de nacimiento y edad...","SELECT name, born_state, age FROM head ORDER..."
2,department_management,"Enumere el año de creación, el nombre y el pre...","SELECT creation, name, budget_in_billions FR..."
3,department_management,¿Cuáles son el presupuesto máximo y mínimo de ...,"SELECT max(budget_in_billions), min(budget_in..."
4,department_management,¿Cuál es el número medio de empleados de los d...,SELECT avg(num_employees) FROM department WHER...


In [29]:
df.shape

(9159, 3)

### Exportación de datos

In [30]:
df.to_csv(f'{datos_limpios}datos_preliminares.csv', index=False)